# Milestone 5: Model Explainability with SHAP

This notebook implements feature attribution and local explanations for the headline HistGradientBoosting model (`core + talent_reach`, V2 static priors).

### Workflow Steps:
1. Explainer compatibility & runtime benchmark (`TreeExplainer` vs. `PermutationExplainer`)
2. Probability-space SHAP additivity verification on the High tier
3. Global feature attribution via Net Group Effects
4. Missingness dynamics & counterfactual interventions
5. Faithfulness benchmark
6. Full local view case studies
7. Standalone explanation utility (`src/explain.py`)

In [1]:
import os
import sys
from pathlib import Path

# Ensure repository root is working directory
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import shap
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

from src.modeling import load_modeling_data
from src.features import FEATURE_GROUPS

print(f'Working directory: {Path.cwd()}')
print(f'Python executable: {sys.executable}')
print(f'SHAP version: {shap.__version__}')


Working directory: C:\Users\DELL\Downloads\Drama Success Copilot
Python executable: C:\Users\DELL\Downloads\Drama Success Copilot\venv\Scripts\python.exe
SHAP version: 0.51.0


## Step 1: Explainer Compatibility & Benchmarking

TreeExplainer does not support multi-output class probabilities for this model, so a model-agnostic PermutationExplainer was used (background: 100 train dramas, 300 test dramas) wrapping `P(High)`.

In [2]:
# Load data and prepare headline features
train_df, test_df = load_modeling_data('v2')
features = FEATURE_GROUPS['core'] + FEATURE_GROUPS['talent_reach']

X_train = train_df[features].copy()
y_train = train_df['target_tier']
X_test = test_df[features].copy()
y_test = test_df['target_tier']

cat_cols = [c for c in ['country', 'original_network', 'content_rating'] if c in features]
cat_indices = [features.index(c) for c in cat_cols]

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = enc.fit_transform(X_train[cat_cols].astype(str))
X_test[cat_cols] = enc.transform(X_test[cat_cols].astype(str))

X_train_num = X_train.astype(np.float64)
X_test_num = X_test.astype(np.float64)

clf = HistGradientBoostingClassifier(
    max_iter=100, learning_rate=0.05, max_leaf_nodes=15,
    min_samples_leaf=30, l2_regularization=1.0, categorical_features=cat_indices,
    random_state=42
)
clf.fit(X_train_num, y_train)
print('Model fitted successfully. Classes:', clf.classes_)

Model fitted successfully. Classes: ['High' 'Low' 'Medium']


In [3]:
# PermutationExplainer benchmark
high_idx = list(clf.classes_).index('High')
predict_high_proba = lambda x: clf.predict_proba(x)[:, high_idx]

bg_100 = X_train_num.sample(n=100, random_state=42)
test_300 = X_test_num.sample(n=300, random_state=42)

cache_path = Path('models/shap_sample_300.npz')
cached = np.load(cache_path)
sv_300_values = cached['values']
test_300_indices = cached['test_indices']
base_val_300 = float(cached['base_values'][0])
runtime = 82.99

print(f'PermutationExplainer runtime for 300 test dramas: {runtime:.2f} s ({runtime/60:.2f} min)')
print(f'Per-sample explanation time: {runtime/300*1000:.1f} ms')
print(f'SHAP values shape: {sv_300_values.shape}')
print(f'Base value (average predicted P(High) over background): {base_val_300:.4f}')

PermutationExplainer runtime for 300 test dramas: 82.99 s (1.38 min)
Per-sample explanation time: 276.6 ms
SHAP values shape: (300, 47)
Base value (average predicted P(High) over background): 0.2494


## Step 2: Probability-Space SHAP Additivity Verification

Additivity holds by construction for `PermutationExplainer`. Checking that base value plus SHAP values equals the raw model output verifies that the explanation pipeline is mathematically consistent (it does not prove the explanations are faithful).

In [4]:
# Verify additivity on 20 random test dramas
sample_20_idx = test_df.sample(n=20, random_state=42).index
sample_20_X = X_test_num.loc[sample_20_idx]

perm_expl_100 = shap.explainers.PermutationExplainer(predict_high_proba, bg_100)
sv_20 = perm_expl_100(sample_20_X)
raw_p_20 = predict_high_proba(sample_20_X)
shap_sum_20 = sv_20.values.sum(axis=1) + sv_20.base_values
abs_errors_20 = np.abs(shap_sum_20 - raw_p_20)

print(f'Sample size: 20 random test dramas')
print(f'Base value (average predicted P(High) over background): {sv_20.base_values[0]:.6f}')
print(f'Max absolute additivity error: {np.max(abs_errors_20):.2e}')
print(f'Mean absolute additivity error: {np.mean(abs_errors_20):.2e}')
print(f'Exact machine precision additivity: {np.max(abs_errors_20) < 1e-12}')

PermutationExplainer explainer:   5%|▌         | 1/20 [00:00<?, ?it/s]

PermutationExplainer explainer:  15%|█▌        | 3/20 [00:12<00:02,  5.68it/s]

PermutationExplainer explainer:  20%|██        | 4/20 [00:12<00:03,  4.23it/s]

PermutationExplainer explainer:  25%|██▌       | 5/20 [00:13<00:04,  3.39it/s]

PermutationExplainer explainer:  30%|███       | 6/20 [00:13<00:04,  2.96it/s]

...

PermutationExplainer explainer:  95%|█████████▌| 19/20 [00:18<00:00,  2.86it/s]

PermutationExplainer explainer: 100%|██████████| 20/20 [00:18<00:00,  2.81it/s]

PermutationExplainer explainer: 21it [00:18,  2.93it/s]                        

PermutationExplainer explainer: 21it [00:18,  1.06it/s]

Sample size: 20 random test dramas
Base value (average predicted P(High) over background): 0.249382
Max absolute additivity error: 1.67e-15
Mean absolute additivity error: 5.95e-16
Exact machine precision additivity: True


## Step 3: Global Feature Attribution via Net Group Effect

Rather than summing $|\text{SHAP}|$ across individual binary features (which artificially inflates groups with many features like `genres`), we compute the **net group effect**: for each drama, we sum the signed SHAP values inside the group, take the absolute value, and average across the 300 dramas.

In [5]:
# 3. Net Group Attribution
group_defs = {
    'talent_cast': ['cast_prior_reach_max', 'cast_prior_reach_mean'],
    'network': ['original_network'],
    'genres': [c for c in features if c.startswith('genre_')],
    'talent_director': ['director_prior_reach_max', 'director_prior_reach_mean'],
    'talent_screenwriter': ['screenwriter_prior_reach_max', 'screenwriter_prior_reach_mean'],
    'content_rating': ['content_rating'],
    'country': ['country'],
    'weekdays': [c for c in features if c.startswith('aired_on_')],
    'episodes': ['episodes'],
}

net_group_shap_300 = {}
abs_net_group_shap_300 = {}

for g, cols in group_defs.items():
    col_indices = [features.index(c) for c in cols]
    net_val = sv_300_values[:, col_indices].sum(axis=1)
    net_group_shap_300[g] = net_val
    abs_net_group_shap_300[g] = np.abs(net_val)

mean_abs_net_shap = {g: float(np.mean(vals)) for g, vals in abs_net_group_shap_300.items()}
mean_signed_shap = {g: float(np.mean(vals)) for g, vals in net_group_shap_300.items()}
sorted_net_groups = sorted(mean_abs_net_shap.items(), key=lambda x: x[1], reverse=True)

df_net_ranking = pd.DataFrame([
    {
        'Feature Group': g,
        'Net Mean |SHAP|': f'{mean_abs_net_shap[g]:.3f}',
        'Mean Signed SHAP': f'{mean_signed_shap[g]:+.3f}',
        'Share (%)': f'{100 * mean_abs_net_shap[g] / sum(mean_abs_net_shap.values()):.1f}%'
    }
    for g, _ in sorted_net_groups
])
print('=== Global Net Group Ranking (300 Test Dramas) ===')
print(df_net_ranking.to_string(index=False))

# Combined talent
talent_combined_net = (
    net_group_shap_300['talent_cast'] +
    net_group_shap_300['talent_screenwriter'] +
    net_group_shap_300['talent_director']
)
print(f'\nCombined Talent (cast + writer + director): Net Mean |SHAP| = {float(np.mean(np.abs(talent_combined_net))):.3f}')

# Generate and save plot to docs/shap_group_importance.png
docs_dir = Path('docs')
docs_dir.mkdir(exist_ok=True)
chart_path = docs_dir / 'shap_group_importance.png'

plt.figure(figsize=(8, 4.5), dpi=150)
groups_plot = [x[0] for x in sorted_net_groups][::-1]
values_plot = [x[1] for x in sorted_net_groups][::-1]
colors = ['#4C72B0' if not g.startswith('talent') else '#DD8452' for g in groups_plot]

bars = plt.barh(groups_plot, values_plot, color=colors, height=0.65)
plt.xlabel('Net Mean |SHAP| (Probability of High Class)', fontsize=10, weight='bold')
plt.title('Headline Model Net Feature Group Attribution (Permutation SHAP, N=300)', fontsize=11, weight='bold', pad=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.001, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.savefig(chart_path)
plt.close()
print(f'Saved bar chart to {chart_path}')

=== Global Net Group Ranking (300 Test Dramas) ===
      Feature Group Net Mean |SHAP| Mean Signed SHAP Share (%)
        talent_cast           0.085           +0.022     19.2%
            network           0.078           +0.007     17.7%
     content_rating           0.073           +0.019     16.7%
             genres           0.065           -0.011     14.7%
talent_screenwriter           0.033           +0.018      7.4%
            country           0.032           -0.012      7.2%
    talent_director           0.032           +0.012      7.2%
           weekdays           0.027           +0.009      6.0%
           episodes           0.017           -0.006      3.9%

Combined Talent (cast + writer + director): Net Mean |SHAP| = 0.118


Saved bar chart to docs\shap_group_importance.png


In [6]:
# 3(a) Stability check with 150 background samples and seed 123 (recomputed net group SHAP)
stab_cache_path = Path('models/stab_net_shap.npz')
cached_stab = np.load(stab_cache_path, allow_pickle=True)
abs_net_stab = cached_stab['abs_net_stab'].item()
stab_runtime = float(cached_stab['runtime'])

r_net_orig = [mean_abs_net_shap[g] for g in group_defs.keys()]
r_net_stab = [abs_net_stab[g] for g in group_defs.keys()]
corr_net, pval_net = spearmanr(r_net_orig, r_net_stab)

print('=== 3(a) Stability Check (Net Group SHAP) ===')
print(f'Runtime (150 background on 300 test): {stab_runtime:.2f} s')
print(f'Spearman rank correlation across 9 groups: {corr_net:.4f} (p-value: {pval_net:.2e})')

# 3(c) complete_info ranking (N=135)
test_300_df = test_df.loc[test_300_indices].copy()
ci_mask_300 = (test_300_df['complete_info'] == 1).values
n_ci_300 = int(ci_mask_300.sum())

ci_net_rows = []
for g, cols in group_defs.items():
    vals_ci = abs_net_group_shap_300[g][ci_mask_300]
    ci_net_rows.append((g, float(np.mean(vals_ci))))

ci_net_rows = sorted(ci_net_rows, key=lambda x: x[1], reverse=True)
df_ci_net = pd.DataFrame([
    {
        'Feature Group': g,
        'Net Mean |SHAP| (complete_info)': f'{val:.3f}',
        'Share (%)': f'{100 * val / sum([x[1] for x in ci_net_rows]):.1f}%'
    }
    for g, val in ci_net_rows
])
print(f'\n=== 3(c) Complete_info Ranking (N = {n_ci_300}) ===')
print(df_ci_net.to_string(index=False))
print('Note: Differences of ~0.01 are within sampling noise; only clearly separated tiers can be claimed.')

=== 3(a) Stability Check (Net Group SHAP) ===
Runtime (150 background on 300 test): 104.46 s
Spearman rank correlation across 9 groups: 1.0000 (p-value: 0.00e+00)

=== 3(c) Complete_info Ranking (N = 135) ===
      Feature Group Net Mean |SHAP| (complete_info) Share (%)
        talent_cast                           0.119     22.0%
     content_rating                           0.086     15.9%
            network                           0.085     15.6%
             genres                           0.070     12.9%
talent_screenwriter                           0.055     10.3%
    talent_director                           0.047      8.6%
            country                           0.034      6.3%
           weekdays                           0.025      4.6%
           episodes                           0.021      3.9%
Note: Differences of ~0.01 are within sampling noise; only clearly separated tiers can be claimed.


## Step 4: Missingness Attribution & Counterfactual Interventions

- A SHAP value near zero for a NaN value does not imply the model treats missing talent as neutral; rather, NaN is the typical background value (~68% director, ~75% writer in train).
- We run counterfactual interventions on complete-information dramas to test the causal effect of setting features to missing.

In [7]:
# Missingness split with signed SHAP and counterfactual tests
missing_check_map = {
    'network': test_300_df['original_network'].isna(),
    'country': pd.Series(False, index=test_300_df.index),
    'genres': test_300_df['genre_is_missing'] == 1,
    'content_rating': test_300_df['content_rating_is_missing'] == 1,
    'weekdays': test_300_df['aired_on_is_missing'] == 1,
    'episodes': pd.Series(False, index=test_300_df.index),
    'talent_director': test_300_df['director_prior_reach_mean'].isna(),
    'talent_screenwriter': test_300_df['screenwriter_prior_reach_mean'].isna(),
    'talent_cast': test_300_df['cast_prior_reach_mean'].isna(),
}

missing_split_rows = []
for g, _ in sorted_net_groups:
    is_miss = missing_check_map[g].values
    n_miss = int(is_miss.sum())
    n_known = int((~is_miss).sum())
    abs_vals = abs_net_group_shap_300[g]
    signed_vals = net_group_shap_300[g]
    mean_abs_k = float(abs_vals[~is_miss].mean()) if n_known > 0 else np.nan
    mean_sgn_k = float(signed_vals[~is_miss].mean()) if n_known > 0 else np.nan
    mean_abs_m = float(abs_vals[is_miss].mean()) if n_miss > 0 else np.nan
    mean_sgn_m = float(signed_vals[is_miss].mean()) if n_miss > 0 else np.nan
    missing_split_rows.append({
        'Feature Group': g,
        'N Known': n_known,
        'Known Mean |SHAP|': f'{mean_abs_k:.3f}' if not np.isnan(mean_abs_k) else 'N/A',
        'Known Signed SHAP': f'{mean_sgn_k:+.3f}' if not np.isnan(mean_sgn_k) else 'N/A',
        'N Missing': n_miss,
        'Missing Mean |SHAP|': f'{mean_abs_m:.3f}' if not np.isnan(mean_abs_m) else 'N/A',
        'Missing Signed SHAP': f'{mean_sgn_m:+.3f}' if not np.isnan(mean_sgn_m) else 'N/A',
    })
df_miss_signed = pd.DataFrame(missing_split_rows)
print('=== Missingness Split (Net |SHAP| and Signed SHAP) ===')
print(df_miss_signed.to_string(index=False))

# Shine (2025) Director & Screenwriter SHAP
shine_mask = test_df['name'].str.contains('Shine', case=False, na=False)
shine_idx = test_df[shine_mask].index[0]
shine_X = X_test_num.loc[[shine_idx]]
sv_shine = perm_expl_100(shine_X)
d_cols_idx = [features.index(c) for c in group_defs['talent_director']]
s_cols_idx = [features.index(c) for c in group_defs['talent_screenwriter']]
print(f'\nShine (2025) Director Net SHAP: {float(sv_shine.values[0, d_cols_idx].sum()):+.6f}')
print(f'Shine (2025) Screenwriter Net SHAP: {float(sv_shine.values[0, s_cols_idx].sum()):+.6f}')

# Counterfactual tests on complete_info test dramas
ci_test_df = test_df[test_df['complete_info'] == 1].copy()
N_ci = len(ci_test_df)
print(f'\n=== Counterfactual Interventions on complete_info (N = {N_ci}) ===')
X_ci_orig = X_test_num.loc[ci_test_df.index].copy()
base_p_high = predict_high_proba(X_ci_orig)
base_preds = clf.predict(X_ci_orig)
base_high_pct = float((base_preds == 'High').mean())

interventions = [
    ('1. Talent Reach (all 6 features) -> NaN', ['director_prior_reach_max', 'director_prior_reach_mean', 'screenwriter_prior_reach_max', 'screenwriter_prior_reach_mean', 'cast_prior_reach_max', 'cast_prior_reach_mean'], np.nan),
    ('2. Broadcaster Network -> NaN (-1)', ['original_network'], -1.0),
    ('3. Content Rating -> NaN (-1)', ['content_rating'], -1.0),
    ('4. Weekdays -> Missing (all 7 = 0)', [c for c in features if c.startswith('aired_on_')], 0.0),
]
cf_results = []
for name, cols, val in interventions:
    X_cf = X_ci_orig.copy()
    X_cf[cols] = val
    cf_p_high = predict_high_proba(X_cf)
    cf_preds = clf.predict(X_cf)
    delta_p = cf_p_high - base_p_high
    mean_delta_p = float(np.mean(delta_p))
    pct_tier_changed = float(np.mean(cf_preds != base_preds)) * 100
    cf_high_pct = float((cf_preds == 'High').mean())
    high_tier_lost = float(np.mean((base_preds == 'High') & (cf_preds != 'High'))) * 100
    cf_results.append({
        'Intervention': name,
        'Mean Delta P(High)': f'{mean_delta_p:+.4f}',
        'Tier Changed (%)': f'{pct_tier_changed:.1f}%',
        'High Tier Lost (%)': f'{high_tier_lost:.1f}%',
        'New High Base Rate': f'{cf_high_pct:.3f} (was {base_high_pct:.3f})',
    })
df_cf = pd.DataFrame(cf_results)
print(df_cf.to_string(index=False))

=== Missingness Split (Net |SHAP| and Signed SHAP) ===
      Feature Group  N Known Known Mean |SHAP| Known Signed SHAP  N Missing Missing Mean |SHAP| Missing Signed SHAP
        talent_cast      214             0.109            +0.022         86               0.023              +0.023
            network      249             0.077            +0.024         51               0.079              -0.079
     content_rating      146             0.097            +0.092        154               0.051              -0.051
             genres      292             0.063            -0.008          8               0.115              -0.115
talent_screenwriter      130             0.066            +0.051        170               0.007              -0.007
            country      300             0.032            -0.012          0                 N/A                 N/A
    talent_director      138             0.053            +0.041        162               0.014              -0.014
           weekda


Shine (2025) Director Net SHAP: -0.015216
Shine (2025) Screenwriter Net SHAP: -0.011481

=== Counterfactual Interventions on complete_info (N = 1022) ===


                           Intervention Mean Delta P(High) Tier Changed (%) High Tier Lost (%) New High Base Rate
1. Talent Reach (all 6 features) -> NaN            -0.1022            31.1%              17.4%  0.334 (was 0.457)
     2. Broadcaster Network -> NaN (-1)            -0.1706            29.7%              23.5%  0.231 (was 0.457)
          3. Content Rating -> NaN (-1)            -0.0958            12.2%              11.8%  0.339 (was 0.457)
     4. Weekdays -> Missing (all 7 = 0)            +0.0011             3.5%               1.3%  0.463 (was 0.457)


## Step 5: Faithfulness Benchmark

For 100 test dramas, we replace the top-attributed feature group with background values and compare the drop in predicted `P(High)` against replacing a random non-top group.

In [8]:
# Faithfulness check (deterministic, 20 random replacements per drama, seed=42)
faith_indices = test_300_indices[:100]
X_faith = X_test_num.loc[faith_indices].copy()
sv_faith_values = sv_300_values[:100]
all_group_names = list(group_defs.keys())
top_drops = []
rand_drops = []
orig_p_faith = predict_high_proba(X_faith)

faith_rng = np.random.RandomState(42)

for i in range(100):
    row_x = X_faith.iloc[i:i+1].copy()
    row_sv = sv_faith_values[i]
    p_orig = orig_p_faith[i]
    
    group_mags = {}
    for g, cols in group_defs.items():
        col_indices = [features.index(c) for c in cols]
        group_mags[g] = abs(float(row_sv[col_indices].sum()))
    top_g = max(group_mags.items(), key=lambda x: x[1])[0]
    other_groups = [g for g in all_group_names if g != top_g]
    
    # Top-attributed group replacement
    top_cols = group_defs[top_g]
    X_rep_top = pd.concat([row_x] * len(bg_100), ignore_index=True)
    X_rep_top[top_cols] = bg_100[top_cols].values
    p_top_marginal = float(np.mean(predict_high_proba(X_rep_top)))
    top_drops.append(abs(p_orig - p_top_marginal))
    
    # Average over 20 random non-top groups per drama
    sampled_rand_groups = faith_rng.choice(other_groups, size=20, replace=True)
    drama_rand_drops = []
    for rand_g in sampled_rand_groups:
        rand_cols = group_defs[rand_g]
        X_rep_rand = pd.concat([row_x] * len(bg_100), ignore_index=True)
        X_rep_rand[rand_cols] = bg_100[rand_cols].values
        p_rand_marginal = float(np.mean(predict_high_proba(X_rep_rand)))
        drama_rand_drops.append(abs(p_orig - p_rand_marginal))
    rand_drops.append(float(np.mean(drama_rand_drops)))

top_drops = np.array(top_drops)
rand_drops = np.array(rand_drops)
mean_top_drop = float(np.mean(top_drops))
mean_rand_drop = float(np.mean(rand_drops))
point_ratio = mean_top_drop / mean_rand_drop

# Bootstrap 95% CI (1000 resamples)
boot_ratios = []
boot_rng = np.random.RandomState(42)
n_faith = len(top_drops)
for _ in range(1000):
    b_idx = boot_rng.choice(n_faith, size=n_faith, replace=True)
    boot_ratios.append(np.mean(top_drops[b_idx]) / np.mean(rand_drops[b_idx]))
ci_low = float(np.percentile(boot_ratios, 2.5))
ci_high = float(np.percentile(boot_ratios, 97.5))

print('=== Faithfulness Check (100 Test Dramas) ===')
print(f'Mean prediction change when replacing TOP-attributed group:    {mean_top_drop:.4f}')
print(f'Mean prediction change when replacing RANDOM non-top group:   {mean_rand_drop:.4f}')
print(f'Ratio of impact (Top vs Random): {point_ratio:.2f}x [95% CI: {ci_low:.2f}x, {ci_high:.2f}x]')


=== Faithfulness Check (100 Test Dramas) ===
Mean prediction change when replacing TOP-attributed group:    0.1610
Mean prediction change when replacing RANDOM non-top group:   0.0386
Ratio of impact (Top vs Random): 4.17x [95% CI: 3.71x, 4.81x]


## Step 6: Full Local View Case Studies (All 9 Signed Groups)

For each case, we report every group's signed attribution so the sum matches `P(High) - base_value` exactly.

In [9]:
import joblib
# Local Case Studies (All 9 Signed Groups with Raw, Calibrated, and Likelihood Band)
test_df_full = test_df.copy()
test_df_full['pred_tier'] = clf.predict(X_test_num)
test_df_full['prob_high'] = predict_high_proba(X_test_num)
iso_cal = joblib.load('models/isotonic_calibrator.joblib')
test_df_full['cal_prob_high'] = iso_cal.predict(test_df_full['prob_high'])

ci_strict_mask = (
    (test_df_full['complete_info'] == 1) &
    (test_df_full['content_rating_is_missing'] == 0) &
    (test_df_full['aired_on_is_missing'] == 0) &
    (test_df_full['genre_is_missing'] == 0) &
    (test_df_full['original_network'].notna()) &
    (test_df_full['director_prior_reach_mean'].notna()) &
    (test_df_full['screenwriter_prior_reach_mean'].notna()) &
    (test_df_full['cast_prior_reach_mean'].notna())
)
ci_strict_df = test_df_full[ci_strict_mask]

tp_idx = ci_strict_df[(ci_strict_df['target_tier'] == 'High') & (ci_strict_df['pred_tier'] == 'High')].sort_values('prob_high', ascending=False).index[0]
tn_idx = ci_strict_df[(ci_strict_df['target_tier'] == 'Low') & (ci_strict_df['pred_tier'] == 'Low')].sort_values('prob_high', ascending=True).index[0]
high_fn = ci_strict_df[(ci_strict_df['target_tier'] == 'High') & (ci_strict_df['pred_tier'] != 'High')].sort_values('prob_high', ascending=True)
wrong_idx = high_fn.index[0]
wrong_type = f'False Negative (Actual High [{high_fn.iloc[0]["target_pct"]:.1f}%], Predicted {high_fn.iloc[0]["pred_tier"]}, P(High)={high_fn.iloc[0]["prob_high"]:.4f})'

miss_idx = test_df_full[
    (test_df_full['complete_info'] == 1) &
    test_df_full['director_prior_reach_mean'].isna() &
    test_df_full['screenwriter_prior_reach_mean'].isna() &
    test_df_full['cast_prior_reach_mean'].notna()
].index[0]

case_indices = [tp_idx, tn_idx, wrong_idx, miss_idx]
case_labels = [
    'Case 1: True Positive (Correct High)',
    'Case 2: True Negative (Correct Low)',
    f'Case 3: Clearly Wrong ({wrong_type})',
    'Case 4: Listed credits with no earlier dramas found (debut or no history)'
]

sv_cases = perm_expl_100(X_test_num.loc[case_indices])
clean_df = pd.read_csv('data/processed/dramas_clean.csv').set_index('name')

def ordinal(n):
    if 11 <= (n % 100) <= 13:
        suffix = 'th'
    else:
        suffix = {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')
    return f"{n}{suffix}"

def describe_group_value(g_name, d_row, c_row):
    if g_name == 'network':
        net = c_row.get('original_network', 'Unknown') if pd.notna(c_row.get('original_network')) else 'Missing'
        return f'Broadcaster: {net}'
    elif g_name == 'country':
        return f'Country: {d_row["country"]}'
    elif g_name == 'genres':
        g_list = c_row.get('genres', 'None') if pd.notna(c_row.get('genres')) else 'Missing'
        return f'Genres: {g_list}'
    elif g_name == 'content_rating':
        cr = c_row.get('content_rating', 'Unknown') if pd.notna(c_row.get('content_rating')) else 'Missing'
        return f'Rating: {cr}'
    elif g_name == 'weekdays':
        ao = c_row.get('aired_on', 'Unknown') if pd.notna(c_row.get('aired_on')) else 'Missing'
        return f'Schedule: {ao}'
    elif g_name == 'episodes':
        return f'Episodes: {int(d_row["episodes"])}'
    elif g_name == 'talent_director':
        val = d_row['director_prior_reach_mean']
        if pd.notna(val):
            val_str = f'director whose earlier dramas averaged around the {ordinal(round(val*100))} percentile of audience reach within country-year'
        else:
            val_str = 'director with no qualifying earlier dramas (Missing history (NaN))'
        return f'Director: {val_str}'
    elif g_name == 'talent_screenwriter':
        val = d_row['screenwriter_prior_reach_mean']
        if pd.notna(val):
            val_str = f'screenwriter whose earlier dramas averaged around the {ordinal(round(val*100))} percentile of audience reach within country-year'
        else:
            val_str = 'screenwriter with no qualifying earlier dramas (Missing history (NaN))'
        return f'Screenwriter: {val_str}'
    elif g_name == 'talent_cast':
        val = d_row['cast_prior_reach_mean']
        if pd.notna(val):
            val_str = f'lead cast whose earlier dramas averaged around the {ordinal(round(val*100))} percentile of audience reach within country-year'
        else:
            val_str = 'lead cast with no qualifying earlier dramas (Missing history (NaN))'
        return f'Lead Cast: {val_str}'
    return g_name

for i, (idx, label) in enumerate(zip(case_indices, case_labels)):
    d_row = test_df_full.loc[idx]
    d_name = d_row['name']
    c_row = clean_df.loc[d_name] if d_name in clean_df.index else {}
    if isinstance(c_row, pd.DataFrame):
        c_row = c_row.iloc[0]
    sv_row = sv_cases.values[i]
    p_high = float(d_row['prob_high'])
    cal_p = float(d_row['cal_prob_high'])
    band = 'Unlikely' if cal_p < 0.20 else ('Possible' if cal_p < 0.50 else ('Likely' if cal_p < 0.75 else 'Very likely'))
    base_p = float(sv_cases.base_values[i])
    
    signed_group_shap = {}
    for g, cols in group_defs.items():
        col_indices = [features.index(c) for c in cols]
        signed_group_shap[g] = float(sv_row[col_indices].sum())
    sum_signed = sum(signed_group_shap.values())
    
    net = c_row.get('original_network', 'Unknown') if pd.notna(c_row.get('original_network')) else 'Missing'
    print(f'\n=======================================================')
    print(f'{label}')
    print(f'Title: {d_name} ({int(d_row["release_year"])}) | Country: {d_row["country"]} | Network: {net}')
    print(f'Actual Tier: {d_row["target_tier"]} ({d_row["target_pct"]*100:.1f}% reach) | Predicted: {d_row["pred_tier"]}')
    print(f'Raw Model P(High): {p_high:.4f} | Calibrated P(High): {cal_p:.4f} | Band: {band}')
    print(f'Base value: {base_p:.4f} | Sum of SHAP: {sum_signed:+.4f} | Raw P(High) - Base: {p_high - base_p:+.4f}')
    print('Full 9-Group Signed Attribution Breakdown:')
    sorted_case_groups = sorted(signed_group_shap.items(), key=lambda x: x[1], reverse=True)
    for g, s in sorted_case_groups:
        print(f'  {s:+.4f} | {g:20s} | {describe_group_value(g, d_row, c_row)}')



Case 1: True Positive (Correct High)
Title: Who Rules the World (2022) (2022) | Country: China | Network: Tencent Video
Actual Tier: High (99.4% reach) | Predicted: High
Raw Model P(High): 0.9814 | Calibrated P(High): 1.0000 | Band: Very likely
Base value: 0.2494 | Sum of SHAP: +0.7320 | Raw P(High) - Base: +0.7320
Full 9-Group Signed Attribution Breakdown:
  +0.2485 | talent_cast          | Lead Cast: lead cast whose earlier dramas averaged around the 76th percentile of audience reach within country-year
  +0.0877 | talent_director      | Director: director whose earlier dramas averaged around the 67th percentile of audience reach within country-year
  +0.0859 | content_rating       | Rating: 13+ - Teens 13 or older
  +0.0756 | network              | Broadcaster: Tencent Video
  +0.0749 | talent_screenwriter  | Screenwriter: screenwriter whose earlier dramas averaged around the 69th percentile of audience reach within country-year
  +0.0641 | genres               | Genres: Romance | 

## Step 7: Production Regime Audits (18+ Distribution, Deployment Calibration, Likelihood Bands)

We audit:
1. Content rating `18+` empirical distribution and SHAP attribution ($N = 163$ on test).
2. Calibration metrics in the production pitch regime (`complete_info` and `strict`).
3. Calibration bands (`Unlikely`, `Possible`, `Likely`, `Very likely`) and empirical High rates.

In [10]:
# Audit 1: Content Rating 18+ and Category Distribution
def wilson_ci(k, n, z=1.95996):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z / denom) * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))
    return max(0.0, center - margin), min(1.0, center + margin)

print('=== 1. Content Rating Audit (Train, Test N=163 for 18+, Sample-300 SHAP) ===')
cr_cols = group_defs['content_rating']
cr_indices = [features.index(c) for c in cr_cols]
s300_cr_shap = sv_300_values[:, cr_indices].sum(axis=1)
test_s300 = test_df.iloc[test_300_indices].copy()
test_s300['cr_shap'] = s300_cr_shap

for cat in ['18+', '13+', '15+', 'G']:
    tr_m = train_df['content_rating'] == cat
    tr_n = int(tr_m.sum())
    tr_k = int((train_df.loc[tr_m, 'target_tier'] == 'High').sum())
    tr_r = tr_k / tr_n
    tr_l, tr_u = wilson_ci(tr_k, tr_n)
    
    te_m = test_df['content_rating'] == cat
    te_n = int(te_m.sum())
    te_k = int((test_df.loc[te_m, 'target_tier'] == 'High').sum())
    te_r = te_k / te_n
    te_l, te_u = wilson_ci(te_k, te_n)
    
    s_m = test_s300['content_rating'] == cat
    s_n = int(s_m.sum())
    s_shap = float(test_s300.loc[s_m, 'cr_shap'].mean())
    print(f'{cat:5s} | Train N={tr_n:3d} ({tr_r*100:.1f}% [{tr_l*100:.1f}%, {tr_u*100:.1f}%]) | Test N={te_n:3d} ({te_r*100:.1f}% [{te_l*100:.1f}%, {te_u*100:.1f}%]) | S300 N={s_n:2d} (Mean SHAP = {s_shap:+.4f})')

# Audit 2: Production-Regime Calibration Table
print('\n=== 2. App Production Regime Calibration (complete_info N=1,022 and strict N=635) ===')
y_test_bin = (test_df['target_tier'].values == 'High').astype(int)
m_comp = test_df['complete_info'].values == 1
m_strict = test_df_full['complete_info_strict'].values == 1 if 'complete_info_strict' in test_df_full else ci_strict_mask
test_cal_p = test_df_full['cal_prob_high'].values
test_raw_p = test_df_full['prob_high'].values

def eval_cal_slice(probs, mask):
    p_s = probs[mask]
    y_s = y_test_bin[mask]
    mp = float(p_s.mean())
    ob = float(y_s.mean())
    br = float(np.mean((p_s - y_s)**2))
    bins = np.linspace(0.0, 1.0, 11)
    bin_ids = np.digitize(p_s, bins) - 1
    ece = sum((bin_ids == b).sum() / len(p_s) * abs(p_s[bin_ids == b].mean() - y_s[bin_ids == b].mean()) for b in range(10) if (bin_ids == b).sum() > 0)
    return mp, ob, mp - ob, ece, br

mp_cr, ob_cr, b_cr, ece_cr, br_cr = eval_cal_slice(test_raw_p, m_comp)
mp_ci, ob_ci, b_ci, ece_ci, br_ci = eval_cal_slice(test_cal_p, m_comp)
mp_sr, ob_sr, b_sr, ece_sr, br_sr = eval_cal_slice(test_raw_p, m_strict)
mp_si, ob_si, b_si, ece_si, br_si = eval_cal_slice(test_cal_p, m_strict)

print(f'Headline Raw   | Comp: {mp_cr:.3f} vs {ob_cr:.3f} ({b_cr:+.3f}), ECE={ece_cr:.4f}, Brier={br_cr:.4f} | Strict: {mp_sr:.3f} vs {ob_sr:.3f} ({b_sr:+.3f}), ECE={ece_sr:.4f}, Brier={br_sr:.4f}')
print(f'Headline Iso   | Comp: {mp_ci:.3f} vs {ob_ci:.3f} ({b_ci:+.3f}), ECE={ece_ci:.4f}, Brier={br_ci:.4f} | Strict: {mp_si:.3f} vs {ob_si:.3f} ({b_si:+.3f}), ECE={ece_si:.4f}, Brier={br_si:.4f}')

# Audit 3: Production Likelihood Bands
print('\n=== 3. Production Likelihood Bands ===')
for s_name, mask in [('Full Test (N=2,525)', np.ones(len(test_df), dtype=bool)), ('complete_info (N=1,022)', m_comp), ('strict (N=635)', m_strict)]:
    print(f'--- {s_name} ---')
    p_sub = test_cal_p[mask]
    y_sub = y_test_bin[mask]
    bands = [('Unlikely', 0.00, 0.20), ('Possible', 0.20, 0.50), ('Likely', 0.50, 0.75), ('Very likely', 0.75, 1.0001)]
    for b_name, b_low, b_high in bands:
        m = (p_sub >= b_low) & (p_sub < b_high) if b_high < 1.0 else (p_sub >= b_low) & (p_sub <= 1.0)
        n_b = int(m.sum())
        k_b = int(y_sub[m].sum())
        rate = k_b / n_b if n_b > 0 else 0.0
        ci_l, ci_u = wilson_ci(k_b, n_b)
        print(f'  {b_name:12s}: N={n_b:4d} | Observed High Rate = {rate*100:5.1f}% [{ci_l*100:4.1f}%, {ci_u*100:4.1f}%]')


=== 1. Content Rating Audit (Train, Test N=163 for 18+, Sample-300 SHAP) ===
18+   | Train N= 82 (64.6% [53.8%, 74.1%]) | Test N=163 (60.1% [52.5%, 67.3%]) | S300 N=14 (Mean SHAP = +0.1759)
13+   | Train N=536 (49.8% [45.6%, 54.0%]) | Test N=323 (45.5% [40.2%, 51.0%]) | S300 N=30 (Mean SHAP = +0.1218)
15+   | Train N=820 (44.3% [40.9%, 47.7%]) | Test N=477 (35.8% [31.7%, 40.2%]) | S300 N=66 (Mean SHAP = +0.1146)
G     | Train N=438 (24.7% [20.9%, 28.9%]) | Test N=261 (8.8% [5.9%, 12.9%]) | S300 N=36 (Mean SHAP = -0.0062)

=== 2. App Production Regime Calibration (complete_info N=1,022 and strict N=635) ===
Headline Raw   | Comp: 0.447 vs 0.389 (+0.057), ECE=0.0858, Brier=0.1801 | Strict: 0.524 vs 0.405 (+0.119), ECE=0.1228, Brier=0.1760
Headline Iso   | Comp: 0.370 vs 0.389 (-0.019), ECE=0.0540, Brier=0.1742 | Strict: 0.451 vs 0.405 (+0.046), ECE=0.0702, Brier=0.1564

=== 3. Production Likelihood Bands ===
--- Full Test (N=2,525) ---
  Unlikely    : N=1436 | Observed High Rate =   7.0%

## Step 8: Production Explainer Module (`src/explain.py`)

Demonstrates the standalone `explain_drama` function matching the audited production API.

In [11]:
from src.explain import load_artifacts, explain_drama

artifacts = load_artifacts()
sample_pitch = {
    'title': 'Prospective Fantasy Romance',
    'country': 'China',
    'network': 'Tencent Video',
    'genres': ['Romance', 'Wuxia', 'Fantasy'],
    'content_rating': '13+',
    'weekdays': ['Monday', 'Tuesday', 'Wednesday'],
    'episodes': 40,
    'director': 'Yin Tao',
    'cast': ['Yang Yang', 'Zhao Lu Si'],
    'planned_start_date': '2024-06-01',
}

result = explain_drama(sample_pitch, artifacts=artifacts, seed=42, max_evals=1000)
print('Predicted Tier:', result['predicted_tier'])
print('Calibrated P(High):', result['calibrated_p_high'], f"({result['likelihood_band']})")
print('Top Positive Drivers:')
for d in result['top_positive_groups']:
    print(f"  + {d['group']}: {d['attribution']:+.4f} | {d['reason']}")
print('Top Negative Drivers:')
for d in result['top_negative_groups']:
    print(f"  - {d['group']}: {d['attribution']:+.4f} | {d['reason']}")


Predicted Tier: High
Calibrated P(High): 0.98 (Very likely)
Top Positive Drivers:
  + talent_cast: +0.2215 | Lead cast prior track record averaged around the 83rd percentile within country-year, which raised High likelihood (+0.221).
  + talent_director: +0.1424 | Director earlier works averaged around the 75th percentile within country-year (+0.142).
  + content_rating: +0.1098 | Content rating '13+' raised predicted High likelihood (+0.110).
Top Negative Drivers:
  - talent_screenwriter: -0.0019 | Screenwriter prior track record is unlisted/missing (NaN) (-0.002).
